# P07 — Preprocessing Dataset UTBK LBI

Notebook ini mengolah dua workbook dataset UTBK LBI menjadi:
- `utbk_clean.csv`: salinan bersih seluruh baris pada sheet **Soal**.
- `qg_examples.json`: subset contoh soal pilihan ganda untuk tiga kategori LBI.
- `filtering_log.csv`: keputusan dan alasan setiap soal dipakai atau disisihkan.

**Catatan:** file Excel asli tidak diubah. Seleksi otomatis hanya memeriksa kelengkapan struktur; kebenaran isi soal, kunci, dan pembahasan tetap perlu ditinjau manusia sebelum dipakai sebagai contoh final QG.

## 1. Persiapan dan lokasi file

Letakkan kedua file Excel asli di `data/raw_materials/utbk/` pada root repository. Notebook ini membaca sheet `Soal` saja karena sheet `Petunjuk`, `Contoh`, dan `Daftar` bukan kumpulan soal utama.

In [4]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw_materials" / "utbk"
OUT_DIR = ROOT / "data" / "utbk_processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = [
    RAW_DIR / "BaseSoalLBI.xlsx",
    RAW_DIR / "BaseSoalLBI2.xlsx",
]

print("Root repository:", ROOT.resolve())
print("Folder input:", RAW_DIR.resolve())
print("Folder output:", OUT_DIR.resolve())

Root repository: /content
Folder input: /content/data/raw_materials/utbk
Folder output: /content/data/utbk_processed


## 2. Memuat sheet Soal dari kedua workbook

Pemetaan kolom kedua file sudah diperiksa. Keduanya menggunakan struktur kolom yang sama. `source_file`, `source_sheet`, dan `source_row` ditambahkan hanya pada salinan kerja untuk melacak asal setiap soal.

In [5]:
missing_files = [str(p) for p in SOURCE_FILES if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "File berikut belum ditemukan. Simpan kedua Excel ke data/raw_materials/utbk/:\\n"
        + "\\n".join(missing_files)
    )

frames = []
for file_path in SOURCE_FILES:
    df = pd.read_excel(file_path, sheet_name="Soal")
    df["source_file"] = file_path.name
    df["source_sheet"] = "Soal"
    # Baris Excel dimulai dari 1; baris pertama adalah header.
    df["source_row"] = np.arange(2, len(df) + 2)
    frames.append(df)

raw_df = pd.concat(frames, ignore_index=True, sort=False)
print("Jumlah baris gabungan:", len(raw_df))
print("Jumlah kolom:", len(raw_df.columns))
display(raw_df.head(3))

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Jumlah baris gabungan: 500
Jumlah kolom: 25


,Kode Soal,Nama Subtes,Topik,Tipe Soal,Teks Soal,Gambar Soal,Opsi A - Teks,Opsi A - Gambar,Opsi B - Teks,Opsi B - Gambar,...,Opsi E - Gambar,Kolom Tabel,Kunci Jawaban,Hint,Pembahasan,Gambar Pembahasan,Tingkat Kesulitan,source_file,source_sheet,source_row
0,LBI-001,LBI,Memahami Informasi,pilihan_ganda,Perhatikan Teks Berikut!\n\nHujan turun sejak ...,NaN,kegagalan murid belajar membaca,,pengabdian guru di daerah terpencil yang menem...,,...,NaN,NaN,B,Cari benang merah dari ketiga paragraf.,"Jawaban: B\n\nMeski gaji tertunda, Bu Rini ber...",NaN,Sedang,BaseSoalLBI.xlsx,Soal,2
1,LBI-002,LBI,Memahami Informasi,pilihan_ganda,Perhatikan Teks Berikut!\n\nAwalnya saya menda...,NaN,tata bahasa isyarat tidak memiliki aturan,,ekspresi wajah hanya memperindah percakapan,,...,NaN,NaN,E,Lihat contoh pada kalimat berikutnya: alis ter...,Jawaban: E\n\nContoh alis yang terangkat menun...,NaN,Sedang,BaseSoalLBI.xlsx,Soal,3
2,LBI-003,LBI,Mengevaluasi dan Merefleksi Informasi,pilihan_ganda,Perhatikan Teks Berikut!\n\nPemerintah memberi...,NaN,Halte bus dicat ulang tahun lalu.,,Penumpang bus umumnya berangkat pagi.,,...,NaN,NaN,D,Melemahkan berarti menyerang premis penumpang ...,Jawaban: D\n\nJika bus jarang beroperasi dan k...,NaN,Sulit,BaseSoalLBI.xlsx,Soal,4


## 3. Audit struktur dataset

Periksa nama kolom, jumlah data terisi, dan sampel isi. Kolom yang dipakai untuk preprocessing mengikuti nama kolom aktual pada kedua workbook.

In [6]:
column_audit = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": [str(raw_df[c].dtype) for c in raw_df.columns],
    "non_null": [int(raw_df[c].notna().sum()) for c in raw_df.columns],
    "sample": [raw_df[c].dropna().astype(str).head(2).tolist() for c in raw_df.columns],
})
display(column_audit)

print("Distribusi tipe soal:")
display(raw_df["Tipe Soal"].value_counts(dropna=False).rename_axis("tipe_soal").to_frame("jumlah"))

print("Distribusi kategori LBI:")
display(raw_df["Topik"].value_counts(dropna=False).rename_axis("kategori").to_frame("jumlah"))

,column,dtype,non_null,sample
0,Kode Soal,object,500,"[LBI-001, LBI-002]"
1,Nama Subtes,object,500,"[LBI, LBI]"
2,Topik,object,500,"[Memahami Informasi, Memahami Informasi]"
3,Tipe Soal,object,500,"[pilihan_ganda, pilihan_ganda]"
4,Teks Soal,object,500,[Perhatikan Teks Berikut!\n\nHujan turun sejak...
5,Gambar Soal,object,41,"[LBI/LBI-013.png, LBI/LBI-027.png]"
6,Opsi A - Teks,object,474,"[kegagalan murid belajar membaca, tata bahasa ..."
7,Opsi A - Gambar,object,225,"[ , ]"
8,Opsi B - Teks,object,475,[pengabdian guru di daerah terpencil yang mene...
9,Opsi B - Gambar,object,225,"[ , ]"


Distribusi tipe soal:


,jumlah
tipe_soal,
pilihan_ganda,450
isian_singkat,25
benar_salah,25


Distribusi kategori LBI:


,jumlah
kategori,
Menemukan Informasi,196
Memahami Informasi,153
Mengevaluasi dan Merefleksi Informasi,151


## 4. Konfigurasi kolom dan kategori

Kolom berikut sesuai dengan struktur aktual `BaseSoalLBI.xlsx` dan `BaseSoalLBI2.xlsx`. Tiga kategori target diambil dari kolom `Topik`.

In [14]:
COLUMN_MAP = {
    "question_id": "Kode Soal",
    "category": "Topik",
    "question_type": "Tipe Soal",
    "difficulty": "Tingkat Kesulitan",
    "question": "Teks Soal",
    "answer": "Kunci Jawaban",
    "hint": "Hint",
    "explanation": "Pembahasan",
    "question_image": "Gambar Soal",
    "table_columns": "Kolom Tabel",
    "option_columns": {letter: f"Opsi {letter} - Teks" for letter in "ABCDE"},
    "option_image_columns": {letter: f"Opsi {letter} - Gambar" for letter in "ABCDE"},
    "explanation_image": "Gambar Pembahasan",
}

TARGET_CATEGORIES = [
    "Menemukan Informasi",
    "Memahami Informasi",
    "Mengevaluasi dan Merefleksi Informasi",
]

required_columns = [
    COLUMN_MAP["question_id"], COLUMN_MAP["category"],
    COLUMN_MAP["question_type"], COLUMN_MAP["question"],
    COLUMN_MAP["answer"], COLUMN_MAP["explanation"],
    *COLUMN_MAP["option_columns"].values(),
]
missing_columns = [c for c in required_columns if c not in raw_df.columns]
if missing_columns:
    raise KeyError(f"Kolom wajib tidak ditemukan: {missing_columns}")

print("Pemetaan kolom valid.")
print("Kategori target:", TARGET_CATEGORIES)

Pemetaan kolom valid.
Kategori target: ['Menemukan Informasi', 'Memahami Informasi', 'Mengevaluasi dan Merefleksi Informasi']


## 5. Membersihkan teks dan membentuk skema standar

Pembersihan hanya dilakukan pada DataFrame hasil baca, bukan pada file Excel sumber. Spasi berulang, non-breaking space, dan karakter BOM dirapikan. Isi substantif soal tidak ditulis ulang.

In [8]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("\\u00a0", " ").replace("\\ufeff", "")
    return re.sub(r"\\s+", " ", value).strip()

for col in raw_df.select_dtypes(include="object").columns:
    raw_df[col] = raw_df[col].map(clean_text)

def normalize_answer(value):
    value = clean_text(value).upper()
    match = re.fullmatch(r"\(?([A-E])\)?[.]?", value)
    return match.group(1) if match else ""

records = []
for _, row in raw_df.iterrows():
    options = {
        letter: clean_text(row[COLUMN_MAP["option_columns"][letter]])
        for letter in "ABCDE"
    }
    answer_raw = clean_text(row[COLUMN_MAP["answer"]])
    records.append({
        "question_id": clean_text(row[COLUMN_MAP["question_id"]]),
        "category": clean_text(row[COLUMN_MAP["category"]]),
        "question_type": clean_text(row[COLUMN_MAP["question_type"]]).lower(),
        "difficulty": clean_text(row[COLUMN_MAP["difficulty"]]),
        "question": clean_text(row[COLUMN_MAP["question"]]),
        **{f"option_{k}": options[k] for k in "ABCDE"},
        "answer_raw": answer_raw,
        "answer": normalize_answer(answer_raw),
        "hint": clean_text(row[COLUMN_MAP["hint"]]),
        "explanation": clean_text(row[COLUMN_MAP["explanation"]]),
        "question_image": clean_text(row[COLUMN_MAP["question_image"]]),
        **{
            f"option_{k}_image": clean_text(row[COLUMN_MAP["option_image_columns"][k]])
            for k in "ABCDE"
        },
        "explanation_image": clean_text(row[COLUMN_MAP["explanation_image"]]),
        "table_columns": clean_text(row[COLUMN_MAP["table_columns"]]),
        "source_file": clean_text(row["source_file"]),
        "source_sheet": clean_text(row["source_sheet"]),
        "source_row": int(row["source_row"]),
    })

clean_df = pd.DataFrame(records)
print("Jumlah baris setelah standardisasi:", len(clean_df))
display(clean_df.head(3))

Jumlah baris setelah standardisasi: 500


,question_id,category,question_type,difficulty,question,option_A,option_B,option_C,option_D,option_E,...,option_A_image,option_B_image,option_C_image,option_D_image,option_E_image,explanation_image,table_columns,source_file,source_sheet,source_row
0,LBI-001,Memahami Informasi,pilihan_ganda,Sedang,Perhatikan Teks Berikut!\n\nHujan turun sejak ...,kegagalan murid belajar membaca,pengabdian guru di daerah terpencil yang menem...,kesulitan ekonomi keluarga penyadap karet,bahaya banjir sungai bagi sekolah,perselisihan guru dan suaminya,...,,,,,,,,BaseSoalLBI.xlsx,Soal,2
1,LBI-002,Memahami Informasi,pilihan_ganda,Sedang,Perhatikan Teks Berikut!\n\nAwalnya saya menda...,tata bahasa isyarat tidak memiliki aturan,ekspresi wajah hanya memperindah percakapan,wajah harus selalu datar,ekspresi wajah tidak diperlukan dalam bahasa i...,"ekspresi wajah membawa makna gramatikal, misal...",...,,,,,,,,BaseSoalLBI.xlsx,Soal,3
2,LBI-003,Mengevaluasi dan Merefleksi Informasi,pilihan_ganda,Sulit,Perhatikan Teks Berikut!\n\nPemerintah memberi...,Halte bus dicat ulang tahun lalu.,Penumpang bus umumnya berangkat pagi.,Sopir bus menyukai pekerjaannya.,Sebagian besar bus kota di kota itu beroperasi...,Bus listrik berbunyi lebih senyap.,...,,,,,,,,BaseSoalLBI.xlsx,Soal,4


## 6. Filter kandidat contoh QG

Kriteria filter struktural:
1. Kategori termasuk salah satu dari tiga kategori LBI.
2. Tipe soal `pilihan_ganda`.
3. Teks soal tidak kosong.
4. Kelima opsi A–E terisi dan tidak duplikat.
5. Kunci jawaban tepat satu huruf A–E.
6. Pembahasan tersedia.
7. Tidak memiliki referensi gambar soal atau kolom tabel yang perlu diperiksa secara visual.

Soal yang memerlukan visual/tabel disisihkan dari subset contoh berbasis teks. Soal tersebut tetap disimpan di `utbk_clean.csv`.

In [9]:
def evaluate_record(row):
    reasons = []
    if row["category"] not in TARGET_CATEGORIES:
        reasons.append("kategori_di_luar_tiga_kategori_LBI")
    if row["question_type"] != "pilihan_ganda":
        reasons.append("bukan_pilihan_ganda")
    if not row["question"]:
        reasons.append("teks_soal_kosong")

    options = [row[f"option_{letter}"] for letter in "ABCDE"]
    if any(not value for value in options):
        reasons.append("opsi_A_E_tidak_lengkap")

    normalized_options = [
        re.sub(r"\\s+", " ", value).strip().casefold()
        for value in options if value
    ]
    if len(normalized_options) != len(set(normalized_options)):
        reasons.append("opsi_duplikat")

    if row["answer"] not in list("ABCDE"):
        reasons.append("kunci_bukan_satu_huruf_A_E")
    if not row["explanation"]:
        reasons.append("pembahasan_kosong")
    if row["question_image"] or row["table_columns"]:
        reasons.append("memerlukan_review_visual_atau_tabel")

    return reasons

clean_df["filter_reasons_list"] = clean_df.apply(evaluate_record, axis=1)
clean_df["filter_reasons"] = clean_df["filter_reasons_list"].apply(lambda x: ";".join(x))
clean_df["eligible_qg_example"] = clean_df["filter_reasons_list"].apply(len).eq(0)

candidate_df = clean_df[clean_df["eligible_qg_example"]].copy()

print("Total soal:", len(clean_df))
print("Kandidat lolos filter:", len(candidate_df))
print("Tidak lolos filter:", len(clean_df) - len(candidate_df))
display(candidate_df.groupby("category").size().rename("jumlah_kandidat").to_frame())

Total soal: 500
Kandidat lolos filter: 411
Tidak lolos filter: 89


,jumlah_kandidat
category,
Memahami Informasi,133
Menemukan Informasi,150
Mengevaluasi dan Merefleksi Informasi,128


## 7. Memilih subset contoh QG

Ambil maksimal 3 contoh per kategori secara deterministik berdasarkan urutan pada workbook. Ini adalah **subset awal**. Sebelum contoh dipakai untuk prompt final QG, tinjau manual apakah soal, kunci, pembahasan, dan pengecohnya memang berkualitas.

In [10]:
EXAMPLES_PER_CATEGORY = 3

# Agar contoh tidak hanya berasal dari satu workbook, ambil:
# 1 soal per kategori dari BaseSoalLBI.xlsx dan 2 soal dari BaseSoalLBI2.xlsx.
SOURCE_QUOTA = {
    "BaseSoalLBI.xlsx": 1,
    "BaseSoalLBI2.xlsx": 2,
}

selected_parts = []
for category in TARGET_CATEGORIES:
    category_candidates = candidate_df[candidate_df["category"] == category]
    for source_file, quota in SOURCE_QUOTA.items():
        source_candidates = category_candidates[
            category_candidates["source_file"] == source_file
        ]
        selected_parts.append(source_candidates.head(quota))

selected_df = pd.concat(selected_parts, ignore_index=True).drop_duplicates(
    subset=["question_id"]
)

selected_ids = set(selected_df["question_id"])
print("Jumlah contoh terpilih per kategori:")
display(selected_df.groupby("category").size().rename("jumlah").to_frame())
print("Distribusi sumber:")
display(selected_df.groupby(["category", "source_file"]).size().rename("jumlah").to_frame())

display(selected_df[[
    "question_id", "category", "source_file", "question",
    "option_A", "option_B", "option_C", "option_D", "option_E",
    "answer", "explanation"
]])

Jumlah contoh terpilih per kategori:


,jumlah
category,
Memahami Informasi,3
Menemukan Informasi,3
Mengevaluasi dan Merefleksi Informasi,3


Distribusi sumber:


jumlah
category                              source_file              
Memahami Informasi                    BaseSoalLBI.xlsx        1
                                      BaseSoalLBI2.xlsx       2
Menemukan Informasi                   BaseSoalLBI.xlsx        1
                                      BaseSoalLBI2.xlsx       2
Mengevaluasi dan Merefleksi Informasi BaseSoalLBI.xlsx        1
                                      BaseSoalLBI2.xlsx       2

,question_id,category,source_file,question,option_A,option_B,option_C,option_D,option_E,answer,explanation
0,LBI-004,Menemukan Informasi,BaseSoalLBI.xlsx,Perhatikan Teks Berikut!\n\nTeks 1 (pelaku wis...,2 dari 10,3 dari 10,4 dari 10,5 dari 10,6 dari 10,C,Jawaban: C\n\nTeks 1 menyebut sekitar 4 dari 1...
1,LBI-251,Menemukan Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nSusu merupakan min...,F,N,Be,Cu,Se,D,Soal tersebut menanyakan simbol unsur dari sal...
2,LBI-252,Menemukan Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nSusu merupakan min...,amilum,selulosa,sukrosa,laktosa,vitamin,D,Soal tersebut menanyakan senyawa karbohidrat y...
3,LBI-001,Memahami Informasi,BaseSoalLBI.xlsx,Perhatikan Teks Berikut!\n\nHujan turun sejak ...,kegagalan murid belajar membaca,pengabdian guru di daerah terpencil yang menem...,kesulitan ekonomi keluarga penyadap karet,bahaya banjir sungai bagi sekolah,perselisihan guru dan suaminya,B,"Jawaban: B\n\nMeski gaji tertunda, Bu Rini ber..."
4,LBI-263,Memahami Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nPelaksanaan refere...,Untuk mendapatkan pengakuan internasional terh...,Karena pemerintah Indonesia ingin menghentikan...,Agar masyarakat Timor-Timur dapat memilih oton...,Karena Indonesia didesak oleh dunia internasional,Untuk memutuskan apakah Indonesia tetap memper...,C,Soal tersebut menanyakan alasan utama pemerint...
5,LBI-268,Memahami Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nBoneka Labubu kini...,Cara perancangan metode produksi yang memanfaa...,Cara penjualan dengan kemasan tanpa memberikan...,Tingkat penyebaran popularitas ke banyak negar...,Strategi pemasaran yang mengandalkan produksi ...,Metode penjualan barang kepada konsumen secara...,B,Soal tersebut menanyakan makna istilah blind b...
6,LBI-003,Mengevaluasi dan Merefleksi Informasi,BaseSoalLBI.xlsx,Perhatikan Teks Berikut!\n\nPemerintah memberi...,Halte bus dicat ulang tahun lalu.,Penumpang bus umumnya berangkat pagi.,Sopir bus menyukai pekerjaannya.,Sebagian besar bus kota di kota itu beroperasi...,Bus listrik berbunyi lebih senyap.,D,Jawaban: D\n\nJika bus jarang beroperasi dan k...
7,LBI-264,Mengevaluasi dan Merefleksi Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nPelaksanaan refere...,Keputusan yang dibuat oleh pemerintah Indonesi...,Tuntutan internasional yang mendesak penyelesa...,Seruan dari negara-negara anggota PBB agar Ind...,Pernyataan dukungan negara-negara anggota PBB ...,Seruan dari anggota Dewan Keamanan PBB untuk m...,B,Soal tersebut menanyakan maksud “resolusi-reso...
8,LBI-266,Mengevaluasi dan Merefleksi Informasi,BaseSoalLBI2.xlsx,Perhatikan teks berikut!\n\nPelaksanaan refere...,Meningkatkan stabilitas ekonomi di wilayah Ind...,Membangun citra Indonesia sebagai negara yang ...,Menurunkan konflik dengan negara tetangga yang...,Mengurangi kebutuhan anggaran pertahanan di In...,Mengurangi tekanan dunia internasional terhada...,B,Soal tersebut menanyakan dampak positif keputu...


## 8. Membuat filtering log

Setiap soal mendapat satu keputusan:
- `terpilih_sebagai_contoh_qg`
- `kandidat_lolos_tidak_masuk_subset`
- `disisihkan`

Alasan dicatat agar proses penyaringan dapat ditelusuri.

In [11]:
log_records = []
for _, row in clean_df.iterrows():
    if row["question_id"] in selected_ids:
        decision = "terpilih_sebagai_contoh_qg"
        reason = "terpilih_dari_kandidat_lolos_filter"
    elif not row["eligible_qg_example"]:
        decision = "disisihkan"
        reason = row["filter_reasons"]
    else:
        decision = "kandidat_lolos_tidak_masuk_subset"
        reason = "lolos_filter_tetapi_tidak_dipilih_karena_batas_3_contoh_per_kategori"

    log_records.append({
        "question_id": row["question_id"],
        "category": row["category"],
        "question_type": row["question_type"],
        "source_file": row["source_file"],
        "source_sheet": row["source_sheet"],
        "source_row": row["source_row"],
        "eligible_qg_example": bool(row["eligible_qg_example"]),
        "decision": decision,
        "reason": reason,
    })

filter_log = pd.DataFrame(log_records)
display(filter_log["decision"].value_counts().rename("jumlah").to_frame())
display(filter_log.head(10))

,jumlah
decision,
kandidat_lolos_tidak_masuk_subset,402
disisihkan,89
terpilih_sebagai_contoh_qg,9


,question_id,category,question_type,source_file,source_sheet,source_row,eligible_qg_example,decision,reason
0,LBI-001,Memahami Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,2,True,terpilih_sebagai_contoh_qg,terpilih_dari_kandidat_lolos_filter
1,LBI-002,Memahami Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,3,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
2,LBI-003,Mengevaluasi dan Merefleksi Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,4,True,terpilih_sebagai_contoh_qg,terpilih_dari_kandidat_lolos_filter
3,LBI-004,Menemukan Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,5,True,terpilih_sebagai_contoh_qg,terpilih_dari_kandidat_lolos_filter
4,LBI-005,Menemukan Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,6,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
5,LBI-006,Mengevaluasi dan Merefleksi Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,7,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
6,LBI-007,Menemukan Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,8,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
7,LBI-008,Memahami Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,9,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
8,LBI-009,Menemukan Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,10,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...
9,LBI-010,Memahami Informasi,pilihan_ganda,BaseSoalLBI.xlsx,Soal,11,True,kandidat_lolos_tidak_masuk_subset,lolos_filter_tetapi_tidak_dipilih_karena_batas...


## 9. Ekspor tiga file hasil P07

`utbk_clean.csv` menyimpan seluruh 500 soal dalam skema standar. `qg_examples.json` menyimpan subset 9 contoh (3 per kategori). `filtering_log.csv` menyimpan keputusan untuk seluruh soal.

In [12]:
# Salinan bersih seluruh soal
clean_export = clean_df.drop(columns=["filter_reasons_list"])
clean_export.to_csv(OUT_DIR / "utbk_clean.csv", index=False, encoding="utf-8-sig")

# Subset contoh QG
examples = []
for _, row in selected_df.iterrows():
    examples.append({
        "question_id": row["question_id"],
        "category": row["category"],
        "question_type": row["question_type"],
        "difficulty": row["difficulty"],
        "question": row["question"],
        "options": {letter: row[f"option_{letter}"] for letter in "ABCDE"},
        "answer": row["answer"],
        "explanation": row["explanation"],
        "hint": row["hint"],
        "source_file": row["source_file"],
        "source_sheet": row["source_sheet"],
        "source_row": int(row["source_row"]),
    })

with open(OUT_DIR / "qg_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)

# Log penyaringan seluruh soal
filter_log.to_csv(OUT_DIR / "filtering_log.csv", index=False, encoding="utf-8-sig")

print("File berhasil disimpan:")
for filename in ["utbk_clean.csv", "qg_examples.json", "filtering_log.csv"]:
    path = OUT_DIR / filename
    print(f"- {path} ({path.stat().st_size:,} bytes)")

File berhasil disimpan:
- /content/data/utbk_processed/utbk_clean.csv (1,063,765 bytes)
- /content/data/utbk_processed/qg_examples.json (25,000 bytes)
- /content/data/utbk_processed/filtering_log.csv (86,628 bytes)


## 10. Validasi hasil ekspor

Validasi ini memastikan jumlah baris, jumlah contoh, kategori, opsi A–E, dan format kunci sesuai aturan subset. Validasi otomatis ini tidak menggantikan pemeriksaan kualitas isi oleh manusia.

In [13]:
with open(OUT_DIR / "qg_examples.json", encoding="utf-8") as f:
    saved_examples = json.load(f)

assert len(clean_export) == 500, "Jumlah baris hasil bersih tidak sesuai dengan data sumber."
assert len(filter_log) == len(clean_export), "Log harus mencakup seluruh soal."
assert len(saved_examples) == EXAMPLES_PER_CATEGORY * len(TARGET_CATEGORIES)

for item in saved_examples:
    assert set(item["options"].keys()) == set("ABCDE")
    assert item["answer"] in list("ABCDE")
    assert item["category"] in TARGET_CATEGORIES
    assert all(item["options"][k].strip() for k in "ABCDE")

distribution = pd.Series([x["category"] for x in saved_examples]).value_counts()
print("Validasi ekspor berhasil.")
print("Total data bersih:", len(clean_export))
print("Kandidat lolos filter:", len(candidate_df))
print("Contoh QG tersimpan:", len(saved_examples))
display(distribution.rename("jumlah_contoh").to_frame())

Validasi ekspor berhasil.
Total data bersih: 500
Kandidat lolos filter: 411
Contoh QG tersimpan: 9


,jumlah_contoh
Menemukan Informasi,3
Memahami Informasi,3
Mengevaluasi dan Merefleksi Informasi,3


## 11. Catatan penggunaan hasil P07

- Gunakan `qg_examples.json` sebagai contoh UTBK pada tahap P14.
- Jangan memasukkan seluruh 411 kandidat ke setiap prompt. Ambil contoh sesuai kategori yang sedang dibuat.
- `utbk_clean.csv` merupakan salinan standar, bukan pengganti workbook asli.
- Periksa manual 9 contoh terpilih sebelum dipakai sebagai few-shot examples final.
- Soal yang ditandai `memerlukan_review_visual_atau_tabel` tidak masuk subset berbasis teks, tetapi tetap tersedia di data bersih.